# Fitbit — Data Exploration

Queries Fitbit parquet tables directly from R2 using DuckDB.

Tables: `fitbit_calories`, `fitbit_steps`, `fitbit_sleep`, `fitbit_exercise`

In [5]:
import duckdb
from dotenv import dotenv_values

env = dotenv_values("../.env")

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")

endpoint = (
    env["R2_ENDPOINT_URL"]
    .removeprefix("https://")
    .removeprefix("http://")
    .rstrip("/")
)
use_ssl = env["R2_ENDPOINT_URL"].startswith("https://")

conn.execute(f"""
    SET s3_endpoint = '{endpoint}';
    SET s3_access_key_id = '{env["R2_ACCESS_KEY_ID"]}';
    SET s3_secret_access_key = '{env["R2_SECRET_ACCESS_KEY"]}';
    SET s3_region = 'auto';
    SET s3_url_style = 'path';
    SET s3_use_ssl = {str(use_ssl).lower()};
""")

## Extract

Runs the fitbit extractor against R2 — processes any ZIPs in the inbox and updates the parquet tables.

In [6]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from pipeline.common.config import PipelineConfig
from pipeline.common.r2 import make_client
from pipeline.extract.fitbit import extract_fitbit

config = PipelineConfig.load("../config/config.yaml")
r2 = make_client(config)
extract_fitbit(r2, config)

[fitbit] archived 1 file(s)
[fitbit/fitbit_calories] 1747891 rows
[fitbit/fitbit_exercise] 0 rows
[fitbit/fitbit_sleep] 969 rows
[fitbit/fitbit_steps] 532160 rows


## Parse local ZIP

Drop a Fitbit Google Takeout ZIP into `notebooks/data/` and run the cells below to inspect each metric without hitting R2.

In [1]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from pipeline.extract.fitbit import _parse_zip, _CALORIES_RE, _EXERCISE_RE, _SLEEP_RE, _STEPS_RE

zip_files = sorted(Path("data").glob("*.zip"))
print(f"Found {len(zip_files)} ZIP(s)")
for p in zip_files:
    print(" ", p)

Found 1 ZIP(s)
  data/takeout-20260329T182319Z-3-001.zip


In [2]:
# Steps — parse from local ZIP
zip_path = zip_files[0]  # change index if you have multiple ZIPs

steps = _parse_zip(zip_path, _STEPS_RE, "dateTime", "value")
print(f"{len(steps)} rows")
steps.tail(10)

532160 rows


datetime,date,value
datetime[μs],date,f64
2026-02-07 09:33:00,2026-02-07,28.0
2026-02-07 09:34:00,2026-02-07,0.0
2026-02-07 09:35:00,2026-02-07,0.0
2026-02-07 09:36:00,2026-02-07,0.0
2026-02-07 09:37:00,2026-02-07,0.0
2026-02-07 09:38:00,2026-02-07,0.0
2026-02-11 19:22:00,2026-02-11,1.0
2026-02-11 19:23:00,2026-02-11,0.0
2026-02-11 19:24:00,2026-02-11,0.0


## SQL queries on R2 tables

Uses DuckDB to query parquet files directly from R2 via the S3-compatible API.

In [7]:
# Calories
conn.execute("""
    SELECT DATE_TRUNC('month', date) AS month, ROUND(SUM(value)) AS total_kcal
    FROM read_parquet('s3://year-in-data/tables/fitbit_calories.parquet')
    GROUP BY month ORDER BY month DESC
    LIMIT 24
""").df()

,month,total_kcal
0,2026-03-01,37.0
1,2026-02-01,36.0
2,2026-01-01,45.0
3,2025-12-01,47.0
4,2025-11-01,40.0
5,2025-10-01,41.0
6,2025-09-01,46.0
7,2025-08-01,41.0
8,2025-07-01,45.0
9,2025-06-01,41.0


In [ ]:
# Steps — raw rows (inspect values look correct)
conn.execute("""
    SELECT datetime, date, value
    FROM read_parquet('s3://year-in-data/tables/fitbit_steps.parquet')
    ORDER BY datetime DESC
    LIMIT 10
""").df()

In [4]:
# Steps
conn.execute("""
    SELECT DATE_TRUNC('month', date) AS month, ROUND(SUM(value)) AS total_steps
    FROM read_parquet('s3://year-in-data/tables/fitbit_steps.parquet')
    GROUP BY month ORDER BY month DESC
    LIMIT 24
""").df()

,month,total_steps
0,2026-02-01,20.0
1,2026-01-01,67.0
2,2025-12-01,104.0
3,2025-11-01,55.0
4,2025-10-01,40.0
5,2025-09-01,37.0
6,2025-08-01,0.0
7,2025-07-01,39.0
8,2025-06-01,24.0
9,2025-05-01,29.0


In [ ]:
# Sleep (hours)
conn.execute("""
    SELECT date, ROUND(value / 60.0, 1) AS hours
    FROM read_parquet('s3://year-in-data/tables/fitbit_sleep.parquet')
    ORDER BY date DESC
    LIMIT 30
""").df()

In [ ]:
# Exercise (active minutes)
conn.execute("""
    SELECT date, ROUND(value / 60000.0, 1) AS active_minutes
    FROM read_parquet('s3://year-in-data/tables/fitbit_exercise.parquet')
    ORDER BY date DESC
    LIMIT 30
""").df()